# Module 05 — Deployment

Test the server locally and prepare for Docker + HF Spaces.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../../src'))
from openenv_env.server import create_app
from fastapi.testclient import TestClient

app = create_app()
client = TestClient(app)
print('Test client ready.')

## Health Check

In [ ]:
r = client.get("/api/v1/health")
print("status:", r.status_code)
print("body  :", r.json())

## Reset + Step + State Roundtrip

In [ ]:
# Reset
r = client.post("/reset", json={})
obs = r.json()
print("reset task_id:", obs["task_id"])
print("reset step   :", obs["step"])

# Step
r = client.post("/step", json={"action": {
    "fixed_code": "def multiply(a, b):\n    return a * b\n",
    "explanation": "Added colon.", "confidence": 0.9
}})
body = r.json()
print("\nstep reward:", body["reward"])
print("step done  :", body["done"])

# State
r = client.get("/state")
state = r.json()
print("\nstate step              :", state["step"])
print("state cumulative_reward :", state["cumulative_reward"])

## Test All Custom v1 Endpoints

In [ ]:
endpoints = [
    ("GET",  "/api/v1/health"),
    ("POST", "/api/v1/reset"),
    ("GET",  "/api/v1/state"),
]
for method, url in endpoints:
    if method == "GET":
        r = client.get(url)
    else:
        r = client.post(url, json={})
    status = "OK" if r.status_code == 200 else "FAIL"
    print(f"  {method:4s}  {url:25s}  {r.status_code}  [{status}]")

## Run Validator (minimal mode)

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "../../validate_submission.py", "--minimal", "--no-docker"],
    capture_output=True, text=True, cwd=os.getcwd()
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

## Generate OpenAPI Schema

In [ ]:
import json
schema = app.openapi()
print("OpenAPI version:", schema["openapi"])
print("Paths:")
for path in schema["paths"]:
    print(" ", path)

## Exercise

Deploy locally:
```bash
python app.py
curl http://localhost:7860/api/v1/health
curl -X POST http://localhost:7860/reset -H 'Content-Type: application/json' -d '{}'
```

Then build and run with Docker:
```bash
docker build -t codedebug-env .
docker run -p 7860:7860 codedebug-env
```